In [7]:
#data spllitting & resizing(already done above)
import os
import random
import shutil
from PIL import Image

# -----------------------------
# SETTINGS
# -----------------------------

input_dir = "C:/Users/pc/Downloads/dataset_converted"        # your original dataset folder
output_dir = "C:/Users/pc/Downloads/dataset_food"           # where processed dataset will go
img_size = (224, 224)            # MobileNetV2 expected size

train_split = 0.70
val_split = 0.20
test_split = 0.10

# -----------------------------
# STEP 1 — Create output folders
# -----------------------------

for split in ["train", "val", "test"]:
    split_path = os.path.join(output_dir, split)
    os.makedirs(split_path, exist_ok=True)

# -----------------------------
# STEP 2 — Process each class
# -----------------------------

classes = os.listdir(input_dir)

for cls in classes:
    class_input_path = os.path.join(input_dir, cls)
    
    if not os.path.isdir(class_input_path):
        continue

    # Create class folders inside train/val/test
    for split in ["train", "val", "test"]:
        os.makedirs(os.path.join(output_dir, split, cls), exist_ok=True)

    # Get image list
    images = [img for img in os.listdir(class_input_path)
              if img.lower().endswith((".jpg", ".jpeg", ".png"))]

    random.shuffle(images)

    # Split sizes
    total = len(images)
    train_end = int(total * train_split)
    val_end = train_end + int(total * val_split)

    train_files = images[:train_end]
    val_files = images[train_end:val_end]
    test_files = images[val_end:]

    # Function to resize + save image
    def process_and_save(img_list, split_name):
        for img_file in img_list:
            img_path = os.path.join(class_input_path, img_file)

            try:
                img = Image.open(img_path).convert("RGB")
                img = img.resize(img_size)

                save_path = os.path.join(output_dir, split_name, cls, img_file)
                img.save(save_path)

            except Exception as e:
                print("Error processing:", img_path, "Error:", e)

    # Process images
    process_and_save(train_files, "train")
    process_and_save(val_files, "val")
    process_and_save(test_files, "test")

print("Dataset processed successfully!")


Dataset processed successfully!


In [8]:
import os
dataset_path="C:/Users/pc/Downloads/dataset_food"
# Define dataset paths
train_path = "C:/Users/pc/Downloads/dataset_food/train"
test_path="C:/Users/pc/Downloads/dataset_food/test"
val_path = "C:/Users/pc/Downloads/dataset_food/val"

# Function to count images per class
def count_images(dataset_path):
    class_counts = {}
    classes = os.listdir(dataset_path)
    for cls in classes:
        class_dir = os.path.join(dataset_path, cls)
        if os.path.isdir(class_dir):
            class_counts[cls] = len(os.listdir(class_dir))
    return class_counts

# Get counts for both sets
train_counts = count_images(train_path)
test_counts=count_images(test_path)
val_counts = count_images(val_path)

# Print class distribution
print("Training Set Distribution:")
for cls, count in train_counts.items():
    print(f"Class: {cls}, Images: {count}")
    
print("\nTest Set Distribution:")
for cls, count in test_counts.items():
    print(f"Class: {cls}, Images: {count}")
    
print("\nValidation Set Distribution:")
for cls, count in val_counts.items():
    print(f"Class: {cls}, Images: {count}")



Training Set Distribution:
Class: apple, Images: 700
Class: banana, Images: 700
Class: chicken_breast, Images: 700
Class: chickpea, Images: 700

Test Set Distribution:
Class: apple, Images: 100
Class: banana, Images: 100
Class: chicken_breast, Images: 100
Class: chickpea, Images: 100

Validation Set Distribution:
Class: apple, Images: 200
Class: banana, Images: 200
Class: chicken_breast, Images: 200
Class: chickpea, Images: 200


In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model

# Paths & Hyperparameters
train_dir = "C:/Users/pc/Downloads/dataset_food/train"
val_dir   = "C:/Users/pc/Downloads/dataset_food/val"
test_dir  = "C:/Users/pc/Downloads/dataset_food/test"  

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 20

#Data Generators (with Strong Augmentation)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    zoom_range=0.25,
    width_shift_range=0.2,
    height_shift_range=0.2,
    brightness_range=[0.8, 1.2],
    horizontal_flip=True,
    shear_range=0.15
)

val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_gen = val_datagen.flow_from_directory(
    val_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_gen = test_datagen.flow_from_directory(
    test_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# Load MobileNetV2 Base Model
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False

# Custom Classification Head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.4)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
preds = Dense(train_gen.num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=preds)

# Compile & Train
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0008),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS
)

# Fine-Tuning
base_model.trainable = True

for layer in base_model.layers[:-35]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_ft = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10
)

# Evaluate on REAL test set
model.evaluate(test_gen)


Found 2800 images belonging to 4 classes.
Found 800 images belonging to 4 classes.
Found 400 images belonging to 4 classes.


C:\Users\pc\anaconda3\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 79s 836ms/step - accuracy: 0.8238 - loss: 0.4941 - val_accuracy: 0.9887 - val_loss: 0.0372
Epoch 2/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 75s 849ms/step - accuracy: 0.9813 - loss: 0.0577 - val_accuracy: 0.9887 - val_loss: 0.0358
Epoch 3/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 75s 850ms/step - accuracy: 0.9857 - loss: 0.0415 - val_accuracy: 0.9875 - val_loss: 0.0380
Epoch 4/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 74s 843ms/step - accuracy: 0.9891 - loss: 0.0439 - val_accuracy: 0.9950 - val_loss: 0.0189
Epoch 5/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 74s 842ms/step - accuracy: 0.9904 - loss: 0.0225 - val_accuracy: 0.9950 - val_loss: 0.0232
Epoch 6/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 74s 842ms/step - accuracy: 0.9889 - loss: 0.0261 - val_accuracy: 0.9925 - val_loss: 0.0312
Epoch 7/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 74s 843ms/step - accuracy: 0.9935 - loss: 0.0193 - val_accuracy: 0.9925 - val_loss: 0.0287
Epoch 8/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 74s 843ms/step - accuracy: 0.9910 - loss: 0.0207 - val_accu

[0.10767897963523865, 0.9900000095367432]

In [2]:
test_gen = val_datagen.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

model.evaluate(test_gen)


Found 400 images belonging to 4 classes.
13/13 ━━━━━━━━━━━━━━━━━━━━ 7s 498ms/step - accuracy: 0.9816 - loss: 0.2150


[0.10767897963523865, 0.9900000095367432]

In [3]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

preds = model.predict(test_gen)
y_pred = np.argmax(preds, axis=1)
y_true = test_gen.classes

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred))


13/13 ━━━━━━━━━━━━━━━━━━━━ 9s 598ms/step
[[ 97   0   3   0]
 [  0 100   0   0]
 [  1   0  99   0]
 [  0   0   0 100]]
              precision    recall  f1-score   support

           0       0.99      0.97      0.98       100
           1       1.00      1.00      1.00       100
           2       0.97      0.99      0.98       100
           3       1.00      1.00      1.00       100

    accuracy                           0.99       400
   macro avg       0.99      0.99      0.99       400
weighted avg       0.99      0.99      0.99       400



In [4]:
#saving model
model.save("mobilenetv2_food_classifier.h5")
print('saved successfully')

saved successfully


In [4]:
import tensorflow as tf
import numpy as np
import cv2


# Load Model
model = tf.keras.models.load_model("C:/Users/Saad_/Downloads/mobilenetv2_food_classifier.h5")

# ----------------------
# 2. Class Names
#    Update this list to YOUR classes (order must match training!)
# ----------------------
class_names = ["apple", "banana","chicken breast", "chickpea"]

# ----------------------
# 3. Load Image
# ----------------------
img_path = "C:/Users/Saad_/Downloads/apple_sample_img.jpeg"   # <-- CHANGE to your image
img = cv2.imread(img_path)

# Convert BGR → RGB
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Resize to the size MobileNetV2 expects (usually 224x224)
img_resized = cv2.resize(img_rgb, (224, 224))

# Normalize & add batch dimension
img_array = img_resized / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Prediction
pred = model.predict(img_array)   
predicted_index = np.argmax(pred)
predicted_category = class_names[predicted_index]


#result
print("Predicted Category:", predicted_category)

# Drawing result on image
cv2.putText(img, predicted_category, (20, 40),
cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)

cv2.imshow("Prediction", img)
cv2.waitKey(0)
cv2.destroyAllWindows()

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
Predicted Category: apple


In [5]:
# Nutrient database (values per 100g)
predicted_weight=200
nutrient_db = {
    "chicken breast": {
        "calories": 165,
        "protein": 31,
        "fat": 3.6,
        "carbohydrates": 0
    },
    "banana": {
        "calories": 89,
        "protein": 1,
        "fat": 0.5,
        "carbohydrates": 23
    },
    "chickpea": {
        "calories": 180,
        "protein": 9.54,
        "fat": 2.99,
        "carbohydrates": 29.98
    },
    "apple": {
        "calories": 51,
        "protein": 0.6,
        "fat": 0.5,
        "carbohydrates": 11.6
    }
}

def calculate_nutrients(food_name, weight_in_grams):
    """
    Calculate nutrients based on recognized food and weight.
    weight_in_grams: weight detected from the scale (in grams)
    """
    
    if food_name not in nutrient_db:
        raise ValueError(f"Nutrient data for '{food_name}' not found.")
    
    food_data = nutrient_db[food_name]

    # scale_factor = weight / 100g
    factor = weight_in_grams / 100

    # multiply each nutrient by factor
    result = {nutrient: value * factor for nutrient, value in food_data.items()}

    return result


# Example usage (you will replace these with your model outputs later):
detected_food = predicted_category
detected_weight = predicted_weight  # e.g., 240 grams detected from scale

nutrients = calculate_nutrients(detected_food, detected_weight)

print(f"{predicted_category} Nutrient Breakdown:")
for k, v in nutrients.items():
    print(f"{k.capitalize()}: {v:.2f}g")


apple Nutrient Breakdown:
Calories: 102.00g
Protein: 1.20g
Fat: 1.00g
Carbohydrates: 23.20g
